In [76]:
import os
from transformers import AutoTokenizer, LlamaForCausalLM
import torch
from peft import PeftModel, PeftConfig

In [77]:
HOME_PATH = os.path.dirname(os.getcwd())
model_path = os.path.join(HOME_PATH, "saved_models", "LlamaFinetuneTestMalky","LlamaFinetune","final_model")

In [78]:
model_path

'/home/home/Desktop/research/saved_models/LlamaFinetuneTestMalky/LlamaFinetune/final_model'

In [79]:
# Debug print to check
print(f"Loading tokenizer from: {model_path}")
assert os.path.exists(model_path), "Model path does not exist!"

Loading tokenizer from: /home/home/Desktop/research/saved_models/LlamaFinetuneTestMalky/LlamaFinetune/final_model


In [80]:
tokenizer = AutoTokenizer.from_pretrained(model_path, local_files_only=True)

In [81]:
tokenizer.special_tokens_map

{'bos_token': '<|begin_of_text|>',
 'eos_token': '<|end_of_text|>',
 'pad_token': '<|end_of_text|>',
 'additional_special_tokens': ['<|user|>', '<|assistant|>']}

In [82]:
original_model_path = os.path.join(HOME_PATH,"Models", 'Llama3.2-1B-Instruct-hf')

In [83]:

model = LlamaForCausalLM.from_pretrained(
            pretrained_model_name_or_path= original_model_path,
            quantization_config={"load_in_8bit": True},  # Pass the quantization config here
            torch_dtype=torch.float16,
            device_map="cuda:0",  # You can uncomment this if you want automatic device allocation
            ignore_mismatched_sizes=True
        )  

# 3. Resize embeddings
model.resize_token_embeddings(len(tokenizer))

# 4. Load the adapter
model = PeftModel.from_pretrained(model, model_path)

In [84]:
#  Build the input
user_prompt = ("A 40-year-old woman undergoing chemotherapy for ovarian carcinoma has developed progressive bilateral sensorineural hearing loss. Which chemotherapy drug is responsible for this side effect?")


full_prompt = f"<|user|> {user_prompt}<|assistant|>"

# Tokenize
inputs = tokenizer(full_prompt, return_tensors="pt").to(model.device)

# Generate
output = model.generate(
    **inputs,
    max_new_tokens=512,
    temperature=0.1,
    top_p=0.9,
    do_sample=True,
    pad_token_id=tokenizer.eos_token_id,
)

In [85]:
# Decode and print
generated_text = tokenizer.decode(output[0], skip_special_tokens=True)
print(generated_text)


# # Decode
# generated_text = tokenizer.decode(output[0], skip_special_tokens=True)
# print(generated_text) 

 A 40-year-old woman undergoing chemotherapy for ovarian carcinoma has developed progressive bilateral sensorineural hearing loss. Which chemotherapy drug is responsible for this side effect?
## Step 1: Identify the type of chemotherapy drug that is known to cause hearing loss.
The chemotherapy drug known to cause hearing loss is typically one that affects the auditory nerve directly.

## Step 2: Recall the specific chemotherapy drugs that are known to cause hearing loss.
Among the common chemotherapy drugs, the ones that are known to cause hearing loss are those that affect the auditory nerve, such as platinum-based drugs.

## Step 3: Identify the specific chemotherapy drug that is known to cause progressive bilateral sensorineural hearing loss.
Given the context of the question, the chemotherapy drug that is known to cause progressive bilateral sensorineural hearing loss is platinum-based, specifically cisplatin.

## Step 4: Confirm the drug's mechanism of action.
Cisplatin works by 

In [59]:
model.config

LlamaConfig {
  "_attn_implementation_autoset": true,
  "architectures": [
    "LlamaForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 128000,
  "eos_token_id": 128001,
  "head_dim": 64,
  "hidden_act": "silu",
  "hidden_size": 2048,
  "initializer_range": 0.02,
  "intermediate_size": 8192,
  "max_position_embeddings": 131072,
  "mlp_bias": false,
  "model_type": "llama",
  "num_attention_heads": 32,
  "num_hidden_layers": 16,
  "num_key_value_heads": 8,
  "pretraining_tp": 1,
  "quantization_config": {
    "_load_in_4bit": false,
    "_load_in_8bit": true,
    "bnb_4bit_compute_dtype": "float32",
    "bnb_4bit_quant_storage": "uint8",
    "bnb_4bit_quant_type": "fp4",
    "bnb_4bit_use_double_quant": false,
    "llm_int8_enable_fp32_cpu_offload": false,
    "llm_int8_has_fp16_weight": false,
    "llm_int8_skip_modules": null,
    "llm_int8_threshold": 6.0,
    "load_in_4bit": false,
    "load_in_8bit": true,
    "quant_method": "bitsandbytes"
 